In [1]:
from dotenv import load_dotenv
from agents import Agent, Runner, trace
from agents.mcp import MCPServerStdio
import os
from IPython.display import Markdown, display
from datetime import datetime
load_dotenv(override=True)

True

In [3]:
import os

os.environ["PATH"] += ":/Users/girisha/.local/bin:/usr/local/bin:/opt/homebrew/bin"

In [4]:
params = {"command": "npx","args": ["-y", "mcp-memory-libsql"],"env": {"LIBSQL_URL": "file:./memory/ed.db"}}

async with MCPServerStdio(params=params, client_session_timeout_seconds=30) as server:
    mcp_tools = await server.list_tools()

mcp_tools

[Tool(name='create_entities', title='Create new entities with observations', description='Create new entities with observations', inputSchema={'type': 'object', 'properties': {'entities': {'type': 'array', 'items': {'type': 'object', 'properties': {'name': {'type': 'string', 'maxLength': 256}, 'entityType': {'type': 'string', 'maxLength': 256}, 'observations': {'type': 'array', 'items': {'type': 'string', 'maxLength': 4096}, 'maxItems': 100}}, 'required': ['name', 'entityType', 'observations']}, 'maxItems': 50}}, 'required': ['entities'], '$schema': 'http://json-schema.org/draft-07/schema#'}, outputSchema=None, icons=None, annotations=ToolAnnotations(title=None, readOnlyHint=False, destructiveHint=None, idempotentHint=True, openWorldHint=None), meta=None),
 Tool(name='search_nodes', title='Search for entities and their relations using text search with relevance ranking', description='Search for entities and their relations using text search with relevance ranking', inputSchema={'type':

In [5]:
instructions = "You use your entity tools as a persistent memory to store and recall information about your conversations."
request = "My name's Ed. I'm an LLM engineer. I'm teaching a course about AI Agents, including the incredible MCP protocol. \
MCP is a protocol for connecting agents with tools, resources and prompt templates, and makes it easy to integrate AI agents with capabilities."
model = "gpt-4.1-mini"

In [6]:
async with MCPServerStdio(params=params, client_session_timeout_seconds=30) as mcp_server:
    agent = Agent(name="agent", instructions=instructions, model=model, mcp_servers=[mcp_server])
    with trace("conversation"):
        result = await Runner.run(agent, request)
    display(Markdown(result.final_output))

Nice to meet you, Ed! It sounds like your course on AI Agents and the MCP protocol is really fascinating. If you need any help preparing materials, examples, or explanations about AI Agents, MCP, or related topics, just let me know. How can I assist you with your course?

In [7]:
async with MCPServerStdio(params=params, client_session_timeout_seconds=30) as mcp_server:
    agent = Agent(name="agent", instructions=instructions, model=model, mcp_servers=[mcp_server])
    with trace("conversation"):
        result = await Runner.run(agent, "My name's Ed. What do you know about me?")
    display(Markdown(result.final_output))

I know that you are Ed, an LLM engineer, and you are teaching a course about AI Agents. The course you teach includes content about the MCP protocol, which is a protocol for connecting agents with tools, resources, and prompt templates. Is there anything specific you'd like me to remember or help you with regarding this information?

### Check the trace:

https://platform.openai.com/traces

### The 2nd type of MCP server - runs locally, calls a web service



In [1]:
import os
os.getenv("SERPER_API_KEY") 

In [16]:
import os

env = {
    **os.environ,
    "SERPER_API_KEY": os.getenv("SERPER_API_KEY"),
}

params = {
    "command": "npx",
    "args": ["-y", "github:marcopesani/mcp-server-serper"],
    "env": env
}

async with MCPServerStdio(params=params, client_session_timeout_seconds=60) as server:
    mcp_tools = await server.list_tools()

mcp_tools

[Tool(name='google_search', title=None, description='Tool to perform web searches via Serper API and retrieve rich results. It is able to retrieve organic search results, people also ask, related searches, and knowledge graph.', inputSchema={'type': 'object', 'properties': {'q': {'type': 'string', 'description': "Search query string (e.g., 'artificial intelligence', 'climate change solutions')"}, 'gl': {'type': 'string', 'description': "Optional region code for search results in ISO 3166-1 alpha-2 format (e.g., 'us', 'gb', 'de')"}, 'hl': {'type': 'string', 'description': "Optional language code for search results in ISO 639-1 format (e.g., 'en', 'es', 'fr')"}, 'location': {'type': 'string', 'description': "Optional location for search results (e.g., 'SoHo, New York, United States', 'California, United States')"}, 'num': {'type': 'number', 'description': 'Number of results to return (default: 10)'}, 'tbs': {'type': 'string', 'description': "Time-based search filter ('qdr:h' for past hou

In [17]:
instructions = "You are able to search the web for information and briefly summarize the takeaways."
request = f"Please research the latest news on Amazon stock price and briefly summarize its outlook. \
For context, the current date is {datetime.now().strftime('%Y-%m-%d')}"
model = "gpt-4o-mini"

In [18]:
async with MCPServerStdio(params=params, client_session_timeout_seconds=30) as mcp_server:
    agent = Agent(name="agent", instructions=instructions, model=model, mcp_servers=[mcp_server])
    with trace("conversation"):
        result = await Runner.run(agent, request)
    display(Markdown(result.final_output))

It seems I'm unable to retrieve specific news articles or updates on Amazon's stock price for June 2026. However, I can provide some general insights into factors that typically impact Amazon's stock outlook, such as market trends, quarterly earnings reports, and economic indicators.

If you're looking for the latest updates, I recommend checking financial news websites like Bloomberg, CNBC, or Yahoo Finance, where you can find real-time stock information and expert analyses.

Would you like to know more about typical factors influencing Amazon's stock price?

### As usual, check out the trace:

https://platform.openai.com/traces

## And the third type: running remotely

It's actually really hard to find a "remote MCP server" aka "hosted MCP server" aka "managed MCP server".




# And back to the 2nd type: the Polygon.io MCP Server

In [20]:
load_dotenv(override=True)
polygon_api_key = os.getenv("POLYGON_API_KEY")
if not polygon_api_key:
    print("POLYGON_API_KEY is not set")

In [21]:
from polygon import RESTClient
client = RESTClient(polygon_api_key)
client.get_previous_close_agg("AAPL")[0]

PreviousCloseAgg(ticker='AAPL', close=291.58, high=294.75, low=287.38, open=290.74, timestamp=1781121600000, volume=52793266.0, vwap=291.412)

In [22]:
from market import get_share_price
get_share_price("AAPL")

291.58

In [23]:
# no rate limiting concerns!

for i in range(1000):
    get_share_price("AAPL")
get_share_price("AAPL")

291.58

In [24]:
params = {"command": "uv", "args": ["run", "market_server.py"]}
async with MCPServerStdio(params=params, client_session_timeout_seconds=60) as server:
    mcp_tools = await server.list_tools()
mcp_tools

[Tool(name='lookup_share_price', title=None, description='This tool provides the current price of the given stock symbol.\n\n    Args:\n        symbol: the symbol of the stock\n    ', inputSchema={'properties': {'symbol': {'title': 'Symbol', 'type': 'string'}}, 'required': ['symbol'], 'title': 'lookup_share_priceArguments', 'type': 'object'}, outputSchema={'properties': {'result': {'title': 'Result', 'type': 'number'}}, 'required': ['result'], 'title': 'lookup_share_priceOutput', 'type': 'object'}, icons=None, annotations=None, meta=None)]

In [25]:
instructions = "You answer questions about the stock market."
request = "What's the share price of Apple?"
model = "gpt-4.1-mini"

async with MCPServerStdio(params=params, client_session_timeout_seconds=60) as mcp_server:
    agent = Agent(name="agent", instructions=instructions, model=model, mcp_servers=[mcp_server])
    with trace("conversation"):
        result = await Runner.run(agent, request)
    display(Markdown(result.final_output))

The current share price of Apple (AAPL) is $291.58.

In [26]:

params = {"command": "uvx",
          "args": ["--from", "git+https://github.com/polygon-io/mcp_polygon@v0.1.0", "mcp_polygon"],
          "env": {"POLYGON_API_KEY": polygon_api_key}
          }
async with MCPServerStdio(params=params, client_session_timeout_seconds=60) as server:
    mcp_tools = await server.list_tools()
mcp_tools


[Tool(name='get_aggs', title=None, description='\n    List aggregate bars for a ticker over a given date range in custom time window sizes.\n    ', inputSchema={'properties': {'ticker': {'title': 'Ticker', 'type': 'string'}, 'multiplier': {'title': 'Multiplier', 'type': 'integer'}, 'timespan': {'title': 'Timespan', 'type': 'string'}, 'from_': {'anyOf': [{'type': 'string'}, {'type': 'integer'}, {'format': 'date-time', 'type': 'string'}, {'format': 'date', 'type': 'string'}], 'title': 'From'}, 'to': {'anyOf': [{'type': 'string'}, {'type': 'integer'}, {'format': 'date-time', 'type': 'string'}, {'format': 'date', 'type': 'string'}], 'title': 'To'}, 'adjusted': {'anyOf': [{'type': 'boolean'}, {'type': 'null'}], 'default': None, 'title': 'Adjusted'}, 'sort': {'anyOf': [{'type': 'string'}, {'type': 'null'}], 'default': None, 'title': 'Sort'}, 'limit': {'anyOf': [{'type': 'integer'}, {'type': 'null'}], 'default': None, 'title': 'Limit'}, 'params': {'anyOf': [{'additionalProperties': True, 'typ

In [27]:
instructions = "You answer questions about the stock market."
request = "What's the share price of Apple? Use your get_snapshot_ticker tool to get the latest price."
model = "gpt-4.1-mini"

async with MCPServerStdio(params=params, client_session_timeout_seconds=60) as mcp_server:
    agent = Agent(name="agent", instructions=instructions, model=model, mcp_servers=[mcp_server])
    with trace("conversation"):
        result = await Runner.run(agent, request)
    display(Markdown(result.final_output))

I currently don't have authorization to access the latest share price of Apple through the snapshot ticker tool. Is there anything else you would like to know or another way I can assist you?